<a href="https://colab.research.google.com/github/Bimorhfdz/f1-podium-prediction-business-insights/blob/main/2_regression_business.ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Project Regresi — Prediksi Poin Driver Formula 1

**Dataset:** Formula 1 World Championship (1950–2024)  
**Source:** [Kaggle - Rohan Rao](https://www.kaggle.com/datasets/rohanrao/formula-1-world-championship-1950-2020)  
**Task:** Regression — Memprediksi jumlah **poin** yang diperoleh seorang driver dalam satu race  

### 📋 Workflow Data Science
1. Business Understanding
2. Data Understanding & Loading
3. Exploratory Data Analysis (EDA)
4. Data Preprocessing
5. Feature Engineering
6. Modeling
7. Evaluation
8. Conclusion & Insight

## 1. Business Understanding

Dalam Formula 1, poin kejuaraan adalah segalanya — menentukan siapa juara dunia di akhir musim.

**Pertanyaan bisnis:**
> *Dapatkah kita memprediksi berapa poin yang akan diperoleh seorang driver dalam satu race, berdasarkan posisi start (grid), performa qualifying, tim (konstruktor), dan faktor lainnya?*

**Tujuan model:**
- Memahami faktor-faktor yang paling berpengaruh terhadap perolehan poin
- Membangun model regresi yang mampu memprediksi poin secara akurat

**Target variabel:** `points` (numerik kontinu: 0, 1, 2, 4, 6, 8, 10, 12, 15, 18, 25)

**Data Quality Control:**
- Filter ke era modern (2015-2024) untuk fair comparison, dan driver aktif
- Alasan: Regulasi stabil (turbo hybrid era), 20 teams kompetitif, eliminasi survivorship bias
- Result: Fokus pada "comparable competitive window" bukan seluruh 75 tahun F1

## 2. Import Library

In [ ]:
# Data Manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold

# Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# Evaluation
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Utilities
import warnings
warnings.filterwarnings('ignore')

# Plot Style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('✅ Semua library berhasil diimport!')

## 3. Data Loading

Dataset diunduh dari Kaggle. Di Google Colab, upload file `results.csv`, `drivers.csv`, dan `constructors.csv` secara manual, atau gunakan Kaggle API.

In [ ]:
# Jalankan cell ini jika belum punya file CSV-nya

!pip install kaggle -q
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d rohanrao/formula-1-world-championship-1950-2020 --unzip

In [ ]:
# Load Dataset
results      = pd.read_csv('results.csv')
drivers      = pd.read_csv('drivers.csv')
constructors = pd.read_csv('constructors.csv')
races        = pd.read_csv('races.csv')
qualifying   = pd.read_csv('qualifying.csv')

print('Shape masing-masing file:')
print(f'  results      : {results.shape}')
print(f'  drivers      : {drivers.shape}')
print(f'  constructors : {constructors.shape}')
print(f'  races        : {races.shape}')
print(f'  qualifying   : {qualifying.shape}')

#  Merge semua tabel
df = results.copy()

# 1. Constructor
df = df.merge(constructors[['constructorId', 'name']], on='constructorId', how='left')
df.rename(columns={'name': 'constructor_name'}, inplace=True)

# 2. Driver (include surname untuk driver_name!)
df = df.merge(drivers[['driverId', 'surname', 'nationality']],
              on='driverId', how='left')
df.rename(columns={'surname': 'driver_name', 'nationality': 'driver_nationality'}, inplace=True)

# 3. Race
df = df.merge(races[['raceId', 'year', 'circuitId']], on='raceId', how='left')

print('\n✅ Merge completed')
print(f'DataFrame shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')

In [ ]:
# Preview dataset utama
print('results.csv (5 baris pertama) ')
display(results.head())

print('\nTipe data & info')
results.info()

## 4. Data Understanding

In [ ]:
# Statistik deskriptif
display(results.describe())

In [ ]:
# Cek missing values
missing = results.isnull().sum()
missing_pct = (missing / len(results) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing (%)': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing (%)', ascending=False)

print('Kolom dengan missing values:')
display(missing_df)

In [ ]:
# Cek duplikat
dup = results.duplicated().sum()
print(f'Jumlah baris duplikat: {dup}')

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# Distribusi target: points
# Ganti '\\N' (string kosong Ergast) dengan 0 lalu convert ke numerik
results['points'] = pd.to_numeric(results['points'], errors='coerce').fillna(0)
results['grid']   = pd.to_numeric(results['grid'],   errors='coerce')
results['positionOrder'] = pd.to_numeric(results['positionOrder'], errors='coerce')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Histogram
axes[0].hist(results['points'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribusi Points per Race Entry')
axes[0].set_xlabel('Points')
axes[0].set_ylabel('Frekuensi')

# Value counts (top 10 poin terbanyak)
vc = results['points'].value_counts().head(10).sort_index()
axes[1].bar(vc.index.astype(str), vc.values, color='salmon', edgecolor='white')
axes[1].set_title('Frekuensi Nilai Poin (Top 10)')
axes[1].set_xlabel('Nilai Poin')
axes[1].set_ylabel('Frekuensi')

plt.tight_layout()
plt.show()

print(f'Rata-rata poin per race entry : {results["points"].mean():.2f}')
print(f'Poin maksimum                 : {results["points"].max()}')
print(f'% Driver yang tidak poin (0)  : {(results["points"]==0).mean()*100:.1f}%')

In [ ]:
# Grid position vs Points
grid_points = results.groupby('grid')['points'].mean().reset_index()
grid_points = grid_points[grid_points['grid'] <= 25]  # filter grid wajar

plt.figure(figsize=(12, 4))
sns.lineplot(data=grid_points, x='grid', y='points', marker='o', color='steelblue')
plt.title('Rata-rata Poin Berdasarkan Posisi Start (Grid)')
plt.xlabel('Grid Position (1 = Pole Position)')
plt.ylabel('Rata-rata Poin')
plt.xticks(range(1, 26))
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 Konstruktor berdasarkan total poin
# Merge dengan nama konstruktor
results_merged = results.merge(
    constructors[['constructorId', 'name']],
    on='constructorId', how='left'
)

top_constructors = (
    results_merged.groupby('name')['points']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(12, 5))
sns.barplot(x=top_constructors.values, y=top_constructors.index, palette='Blues_r')
plt.title('Top 10 Konstruktor — Total Poin Sepanjang Sejarah F1')
plt.xlabel('Total Points')
plt.ylabel('Konstruktor')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap Korelasi
num_cols = ['grid', 'positionOrder', 'points', 'laps', 'milliseconds']

# Convert kolom-kolom ini ke numerik
for col in num_cols:
    results[col] = pd.to_numeric(results[col], errors='coerce')

plt.figure(figsize=(8, 6))
corr_matrix = results[num_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    mask=mask, linewidths=0.5, square=True
)
plt.title('Heatmap Korelasi Fitur Numerik')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: Grid vs Points (dengan warna posisi finish)
sample = results[results['grid'] <= 25].dropna(subset=['grid', 'points', 'positionOrder']).sample(3000, random_state=42)

plt.figure(figsize=(10, 5))
scatter = plt.scatter(
    sample['grid'], sample['points'],
    c=sample['positionOrder'], cmap='RdYlGn_r',
    alpha=0.5, s=20
)
plt.colorbar(scatter, label='Posisi Finish')
plt.title('Hubungan Grid Start vs Poin (warna = posisi finish)')
plt.xlabel('Grid Position')
plt.ylabel('Points')
plt.tight_layout()
plt.show()

## 6. Data Preprocessing & Feature Engineering

In [ ]:
# Merge semua tabel yang diperlukan
df = results.copy()

# Tambahkan nama konstruktor
df = df.merge(constructors[['constructorId', 'name']], on='constructorId', how='left')
df.rename(columns={'name': 'constructor_name'}, inplace=True)

# Tambahkan info driver (nationality)
df = df.merge(drivers[['driverId', 'nationality']], on='driverId', how='left')
df.rename(columns={'nationality': 'driver_nationality'}, inplace=True)

# Tambahkan tahun race
df = df.merge(races[['raceId', 'year', 'circuitId']], on='raceId', how='left')

print(f'Shape setelah merge: {df.shape}')
display(df[['raceId', 'driverId', 'constructor_name', 'grid', 'positionOrder', 'points', 'year']].head())

In [ ]:
# Filter era modern (2015-2024) untuk fair comparison
# Era ini lebih konsisten: regulasi sama, teams lebih stabil
df = df[df['year'] >= 2015].copy()

print(f'Filtered dataset (2015-2024): {len(df)} rows')
print(f'Teams: {df["constructor_name"].nunique()}')
print(f'Drivers: {df["driverId"].nunique()}')
print(f'Seasons: {df["year"].nunique()}')
print()
print('✅ Data quality improved: Eliminated survivorship bias & era effects')

In [ ]:
# Pilih dan bersihkan fitur
# Pastikan driverId terbawa dari hasil merge sebelumnya
df = results.copy()
df = df.merge(constructors[['constructorId', 'name']], on='constructorId', how='left')
df.rename(columns={'name': 'constructor_name'}, inplace=True)
df = df.merge(drivers[['driverId', 'nationality']], on='driverId', how='left')
df.rename(columns={'nationality': 'driver_nationality'}, inplace=True)
df = df.merge(races[['raceId', 'year', 'circuitId']], on='raceId', how='left')

# Filter era modern
df = df[df['year'] >= 2015].copy()

# Sekarang kita bisa menyertakan 'driverId' dengan aman
features_raw = ['driverId', 'grid', 'positionOrder', 'laps', 'constructor_name',
                'driver_nationality', 'year', 'circuitId', 'points']

df = df[features_raw].copy()

# Convert ke numerik
for col in ['grid', 'positionOrder', 'laps', 'points']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop baris yang masih ada NaN
df.dropna(inplace=True)

print(f'Shape setelah cleaning: {df.shape}')
display(df.head())

In [ ]:
# Feature Engineering

# 1. Selisih posisi: berapa banyak posisi yang diperbaiki/hilang dari grid ke finish
df['positions_gained'] = df['grid'] - df['positionOrder']

# 2. Flag apakah start dari top 3 (front row advantage)
df['is_front_row'] = (df['grid'] <= 3).astype(int)

# 3. Flag apakah finish di top 10 (yang mendapat poin di era modern)
df['finished_top10'] = (df['positionOrder'] <= 10).astype(int)

print('Fitur baru berhasil dibuat:')
display(df[['grid', 'positionOrder', 'positions_gained', 'is_front_row', 'finished_top10', 'points']].head(10))

In [ ]:
# Label Encoding untuk fitur kategorikal
le = LabelEncoder()

df['constructor_enc']   = le.fit_transform(df['constructor_name'])
df['nationality_enc']   = le.fit_transform(df['driver_nationality'])

print('Encoding selesai ✅')
print(df[['constructor_name', 'constructor_enc', 'driver_nationality', 'nationality_enc']].drop_duplicates().head(10))

In [ ]:
# Definisi final features & target
FEATURE_COLS = [
    'grid', 'laps', 'year', 'circuitId',
    'positions_gained', 'is_front_row', 'finished_top10',
    'constructor_enc', 'nationality_enc'
]

TARGET = 'points'

X = df[FEATURE_COLS]
y = df[TARGET]

print(f'Shape X: {X.shape}')
print(f'Shape y: {y.shape}')
print(f'\nDistribusi target (points):')
print(y.describe())

In [ ]:
# Train-Test Split (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train size : {X_train.shape[0]} baris')
print(f'Test size  : {X_test.shape[0]} baris')

In [ ]:
# Scaling (untuk model linear)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Scaling selesai ✅')

## 7. Modeling

Kita akan melatih dan membandingkan 5 model regresi:
1. Linear Regression (baseline)
2. Ridge Regression
3. Decision Tree Regressor
4. Random Forest Regressor
5. XGBoost Regressor

In [ ]:
# Training semua model
models = {
    'Linear Regression' : LinearRegression(),
    'Ridge Regression'  : Ridge(alpha=1.0),
    'Decision Tree'     : DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest'     : RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost'           : XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42, verbosity=0)
}

results_dict = {}

for name, model in models.items():
    # Gunakan scaled data untuk model linear, raw untuk tree-based
    if name in ['Linear Regression', 'Ridge Regression']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    results_dict[name] = {'MAE': mae, 'RMSE': rmse, 'R²': r2, 'y_pred': y_pred}
    print(f'[{name}]  MAE={mae:.3f}  RMSE={rmse:.3f}  R²={r2:.4f}')

print('\n✅ Training selesai!')

---
## 8. 📈 Evaluation

In [ ]:
# Tabel perbandingan model
eval_df = pd.DataFrame({
    name: {'MAE': v['MAE'], 'RMSE': v['RMSE'], 'R²': v['R²']}
    for name, v in results_dict.items()
}).T.round(4)

eval_df = eval_df.sort_values('RMSE')

print('Perbandingan Performa Model')
display(eval_df.style.background_gradient(cmap='RdYlGn', subset=['R²'])
                     .background_gradient(cmap='RdYlGn_r', subset=['RMSE', 'MAE']))

In [ ]:
# Visualisasi perbandingan RMSE dan R²
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = eval_df.index.tolist()
rmse_vals   = eval_df['RMSE'].values
r2_vals     = eval_df['R²'].values

# RMSE (lower is better)
bars1 = axes[0].barh(model_names, rmse_vals, color='salmon', edgecolor='white')
axes[0].set_title('RMSE per Model (lebih rendah = lebih baik)')
axes[0].set_xlabel('RMSE')
for bar, val in zip(bars1, rmse_vals):
    axes[0].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)

# R² (higher is better)
bars2 = axes[1].barh(model_names, r2_vals, color='steelblue', edgecolor='white')
axes[1].set_title('R² Score per Model (lebih tinggi = lebih baik)')
axes[1].set_xlabel('R²')
for bar, val in zip(bars2, r2_vals):
    axes[1].text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted — Model Terbaik
best_model_name = eval_df.index[0]  # model dengan RMSE terendah
best_pred = results_dict[best_model_name]['y_pred']

plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_pred, alpha=0.3, s=15, color='steelblue', label='Prediksi')
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', linewidth=2, label='Garis Ideal')
plt.title(f'Actual vs Predicted — {best_model_name}')
plt.xlabel('Actual Points')
plt.ylabel('Predicted Points')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residual Plot — Model Terbaik
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].scatter(best_pred, residuals, alpha=0.3, s=15, color='purple')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title(f'Residual Plot — {best_model_name}')
axes[0].set_xlabel('Predicted Points')
axes[0].set_ylabel('Residual')

axes[1].hist(residuals, bins=40, color='purple', edgecolor='white', alpha=0.7)
axes[1].set_title('Distribusi Residual')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frekuensi')

plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance — Random Forest
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
importances.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Feature Importance — Random Forest Regressor')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('\nFitur paling berpengaruh:')
print(importances.sort_values(ascending=False).head(5))

In [ ]:
# Cross Validation — Model Terbaik
best_model_obj = models[best_model_name]
kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    best_model_obj, X, y,
    cv=kf,
    scoring='neg_root_mean_squared_error'
)
cv_rmse = -cv_scores

print(f'5-Fold Cross Validation — {best_model_name}')
for i, score in enumerate(cv_rmse, 1):
    print(f'  Fold {i}: RMSE = {score:.4f}')
print(f'\n  Mean RMSE : {cv_rmse.mean():.4f}')
print(f'  Std RMSE  : {cv_rmse.std():.4f}')

## 9. Conclusion & Insight

In [ ]:
# Summary Akhir
print('   RINGKASAN PROJECT REGRESI — F1 POINTS PREDICTION')
print(f'\n📌 Dataset    : Formula 1 World Championship (2010–2024)')
print(f'📌 Dataset    : Formula 1 (2015-2024 modern era)')
print(f'📌 Target     : points (poin yang diperoleh per race entry)')
print(f'📌 Teams      : 20 tim kompetitif (fair comparison)')

print(f'\n📊 Performa Model Terbaik: {best_model_name}')
print(f'   - MAE  = {results_dict[best_model_name]["MAE"]:.3f}')
print(f'   - RMSE = {results_dict[best_model_name]["RMSE"]:.3f}')
print(f'   - R²   = {results_dict[best_model_name]["R²"]:.4f}')
print()
print('💡 Insight Utama:')
print('   1. Posisi finish (positionOrder) adalah prediktor terkuat poin.')
print('   2. Grid start sangat berkorelasi dengan poin — pole position')
print('      rata-rata menghasilkan poin tertinggi.')
print('   3. Konstruktor (tim) memengaruhi perolehan poin secara signifikan,')
print('      mencerminkan dominasi tim top seperti Mercedes, Red Bull, Ferrari.')
print('   4. Model berbasis ensemble (RF, XGBoost) jauh lebih baik dari')
print('      linear regression karena distribusi poin yang tidak linear.')

9. Driver Analisis

In [ ]:
# DATA QUALITY: Filter Active Drivers
latest_year = df['year'].max()

# Keep hanya driver yang masih aktif di 2023-2024
df_recent = df[df['year'] >= latest_year - 1].copy()
active_drivers = df_recent['driverId'].unique()

df = df[df['driverId'].isin(active_drivers)].copy()

print(f'✅ Filtered to ACTIVE DRIVERS ONLY')
print(f'   Active Drivers: {df["driverId"].nunique()}')
print(f'   Rows: {len(df)}')
print(f'   Reason: Eliminasi retired drivers (fair comparison)\n')

In [ ]:
# 9. DRIVER ANALYSIS (REGRESI VERSION)

df_with_names = df.merge(drivers[['driverId', 'surname']], on='driverId', how='left')
df_with_names.rename(columns={'surname': 'driver_name'}, inplace=True)

print('✅ Driver names loaded')

# DRIVER PERFORMANCE BY POINTS
print('DRIVER RANKING BY POINTS (REGRESI)')

driver_points = df_with_names.groupby('driver_name').agg({
    'points': ['sum', 'mean', 'std', 'count'],
    'grid': 'mean',
    'positionOrder': 'mean'
}).round(3)

driver_points.columns = ['Total_Points', 'Avg_Points', 'Consistency_Std',
                         'Race_Count', 'Avg_Grid', 'Avg_Finish_Pos']

# Filter: minimal 20 races
driver_points = driver_points[driver_points['Race_Count'] >= 20].copy()

# Performance Score = (avg_points/25 * 0.5) + (grid quality * 0.3) + (consistency * 0.2)
driver_points['Performance_Score'] = (
    (driver_points['Avg_Points'] / 25) * 0.5 +
    (1 - driver_points['Avg_Grid'] / driver_points['Avg_Grid'].max()) * 0.3 +  # lower grid = better
    (1 - driver_points['Consistency_Std'] / driver_points['Consistency_Std'].max()) * 0.2  # lower std = better
).round(3)

driver_ranking = driver_points.sort_values('Performance_Score', ascending=False).head(15)

print('\n📊 Top 15 Drivers by Points:\n')
display(driver_ranking[['Total_Points', 'Avg_Points', 'Race_Count', 'Performance_Score']])

# Visualisasi
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Total Points
ax1 = axes[0, 0]
top_total = driver_ranking.sort_values('Total_Points', ascending=False)
sns.barplot(x=top_total['Total_Points'], y=top_total.index,
            palette='Blues_r', ax=ax1)
ax1.set_title('Total Points (Career 2015-2024)', fontweight='bold')
ax1.set_xlabel('Total Points')

# 2. Average Points per Race
ax2 = axes[0, 1]
top_avg = driver_ranking.sort_values('Avg_Points', ascending=False)
sns.barplot(x=top_avg['Avg_Points'], y=top_avg.index,
            palette='Greens_r', ax=ax2)
ax2.set_title('Average Points Per Race', fontweight='bold')
ax2.set_xlabel('Avg Points')

# 3. Consistency (lower std = better)
ax3 = axes[1, 0]
top_consistency = driver_ranking.sort_values('Consistency_Std', ascending=True)
sns.barplot(x=top_consistency['Consistency_Std'], y=top_consistency.index,
            palette='Oranges_r', ax=ax3)
ax3.set_title('Consistency (Lower = More Stable)', fontweight='bold')
ax3.set_xlabel('Points Std Dev')

# 4. Performance Score (Combined)
ax4 = axes[1, 1]
top_perf = driver_ranking.sort_values('Performance_Score', ascending=False)
sns.barplot(x=top_perf['Performance_Score'], y=top_perf.index,
            palette='viridis', ax=ax4)
ax4.set_title('Overall Performance Score', fontweight='bold')
ax4.set_xlabel('Performance Score')

plt.tight_layout()
plt.show()

# Scatter: Avg Points vs Consistency
fig, ax = plt.subplots(figsize=(12, 7))
scatter = ax.scatter(
    driver_ranking['Avg_Points'],
    1 - (driver_ranking['Consistency_Std'] / driver_ranking['Consistency_Std'].max()),
    s=driver_ranking['Race_Count']*8,
    alpha=0.6,
    c=driver_ranking['Performance_Score'],
    cmap='plasma',
    edgecolor='black',
    linewidth=1.5
)

for name in driver_ranking.index[:15]:
    row = driver_ranking.loc[name]
    ax.annotate(name,
               (row['Avg_Points'], 1 - (row['Consistency_Std'] / driver_ranking['Consistency_Std'].max())),
               fontsize=8, fontweight='bold', ha='center')

ax.set_xlabel('Average Points Per Race (Productivity) →', fontsize=11, fontweight='bold')
ax.set_ylabel('Consistency (Lower variance = better) ↑', fontsize=11, fontweight='bold')
ax.set_title('Driver Performance Analysis (Regresi)\nBubble size = Race count',
            fontsize=13, fontweight='bold')

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Performance Score', fontsize=10)
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print('\n✅ Regresi Driver Analysis Complete!')

### Project Regresi Selesai!

**Summary Kedua Project:**

| Aspek | Regresi | Klasifikasi |
|-------|---------|-------------|
| **Task** | Prediksi nilai poin numerik | Binary: Podium vs Non-Podium |
| **Models** | Linear, Ridge, Tree, RF, XGB | Logistic, DT, RF, GB, XGB, SVM |
| **Metrics** | MAE, RMSE, R² | Accuracy, Precision, Recall, F1, AUC |
| **Class Balance** | N/A | Handled with SMOTE |
| **Best Approach** | Ensemble (RF, XGB) | Ensemble + Probability Calibration |

*Dataset: Formula 1 World Championship | Source: Kaggle (Rohan Rao)*